In [1]:
import pandas as pd
import random

from data_processing.data_processor import concat_nl_and_code

## Consistency Processing

In [19]:
# df = pd.read_csv('../data/Menagerie/data/ungraded_code_grades.csv', index_col=0)
df = pd.read_csv('../data/Menagerie/data/docstring_code_grades.csv', usecols=['assignment_number', 'docstring', 'function', 'grade'])
df.head()

,assignment_number,grade,docstring,function
0,2,A,This is what the rabbit does most of the time ...,public void act(List<Animal> newRabbits)\n ...
1,2,A,Increase the age.\nThis could result in the ra...,private void incrementAge()\n {\n ag...
2,2,A,Make this rabbit more hungry. This could resul...,private void incrementHunger()\n {\n ...
3,2,A,Look for rabbits adjacent to the current locat...,private Location findFood()\n {\n Fi...
4,2,A,Check whether or not this rabbit is to give bi...,private void giveBirth(List<Animal> newRabbits...


In [20]:
grouped_df = df.groupby('assignment_number').count()
# grouped_df.columns = ['count_doc', 'count_func']
grouped_df.columns = ['count_grade', 'count_doc', 'count_func']
grouped_df.head(100)

,count_grade,count_doc,count_func
assignment_number,,,
2,104,104,104
6,98,98,98
8,112,111,112
9,134,134,134
10,122,122,122
...,...,...,...
241,122,120,122
246,118,118,118
249,154,153,154


In [21]:
grouped_df['count_doc'].median()

111.5

In [22]:
WINDOW = 5
median_df = grouped_df[(grouped_df['count_doc'] < grouped_df['count_doc'].median() + WINDOW) &
                       (grouped_df['count_doc'] > grouped_df['count_doc'].median() - WINDOW)]
median_df

,count_grade,count_doc,count_func
assignment_number,,,
8,112,111,112
18,134,111,134
33,112,112,112
36,113,112,113
38,115,115,115
40,146,115,146
45,116,115,116
46,107,107,107
55,111,110,111


In [25]:
SAMPLE_SIZE = grouped_df.shape[0]
random.seed(42)

sample_df = df[df['assignment_number'].isin(grouped_df.sample(SAMPLE_SIZE).index)]
# sample_df.columns = ['assignment_number', 'query', 'func_code_string']
sample_df.columns = ['assignment_number', 'grade', 'query', 'func_code_string']
sample_df = concat_nl_and_code(sample_df)
sample_df = sample_df.dropna()
sample_df

,assignment_number,grade,query,func_code_string,text
0,2,A,This is what the rabbit does most of the time ...,public void act(List<Animal> newRabbits)\n ...,This is what the rabbit does most of the time ...
1,2,A,Increase the age.\nThis could result in the ra...,private void incrementAge()\n {\n ag...,Increase the age.\nThis could result in the ra...
2,2,A,Make this rabbit more hungry. This could resul...,private void incrementHunger()\n {\n ...,Make this rabbit more hungry. This could resul...
3,2,A,Look for rabbits adjacent to the current locat...,private Location findFood()\n {\n Fi...,Look for rabbits adjacent to the current locat...
4,2,A,Check whether or not this rabbit is to give bi...,private void giveBirth(List<Animal> newRabbits...,Check whether or not this rabbit is to give bi...
...,...,...,...,...,...
32801,675,B+,Return the animal's field.\n@return The animal...,protected Field getField()\n {\n ret...,Return the animal's field.\n@return The animal...
32803,675,B+,Check whether the plant is alive or not.\n@ret...,protected boolean isAlive()\n {\n re...,Check whether the plant is alive or not.\n@ret...
32804,675,B+,Indicate that the plant is no longer alive.\nI...,protected void setDead()\n {\n alive...,Indicate that the plant is no longer alive.\nI...
32805,675,B+,Return the plant's location.\n@return The plan...,protected Location getLocation()\n {\n ...,Return the plant's location.\n@return The plan...


In [26]:
sample_df['assignment_number'].unique().shape

(272,)

In [27]:
sample_df.to_csv(f'../data/consistency_sample_{SAMPLE_SIZE}.csv')